In [1]:
import random
random.seed(42)

import torch

import numpy as np
import pyomo.environ as pyo
import pandas as pd
import matplotlib.pyplot as plt

from pyomo.environ import *
from utils import generate_simple_synthetic_data, const_embed, load_case_study_data
from trainer import Trainer

np.random.seed(43)

LOAD_CASE_STUDY = False

In [2]:
if not LOAD_CASE_STUDY:
    X, y, true_mean, true_std = generate_simple_synthetic_data(n_samples=1000)

    print(X.head())
    print(y.head())
else:
    print("Loading case study data...")
    X, y, val_ind, test_ind = load_case_study_data()



         x1        x2      price
0  0.374540  0.185133  15.234114
1  0.950714  0.541901  14.939576
2  0.731994  0.872946  28.125092
3  0.598658  0.732225  14.990924
4  0.156019  0.806561  15.438995
0    4.164115
1    9.161541
2    0.433392
3    5.661846
4    5.218591
Name: y, dtype: float64


In [3]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

'''model_config = {
    'hidden_layers': 1,
    'hidden_size': 5,
    'drop': 0.2
    }

training_config = {
    'lr': 0.01,
    'wd': 0.0001,
    'batchsize': 64,
    'iters': 500,
    'patience': 20
    }'''

config = {
    # Model config
    'hidden_layers': 0,
    'hidden_size': 50,
    'drop': 0.05,

    # Training config
    'lr': 0.001,
    'wd': 0,
    'batchsize': 64,
    'iters': 5000,
    'patience': 1000
}

n_preds = 1

trainer = Trainer(X, y, config, n_preds=n_preds, device=device)

if LOAD_CASE_STUDY:
    trainer.set_data_partitions(X_train=X[(~X.index.isin(val_ind)) & (~X.index.isin(test_ind))], y_train=y[(~y.index.isin(val_ind)) & (~y.index.isin(test_ind))], X_val=X[X.index.isin(val_ind)], y_val=y[y.index.isin(val_ind)], X_test=X[X.index.isin(test_ind)], y_test=y[y.index.isin(test_ind)])
else:
    trainer.create_data_partitions(val_size=0.2, test_size=0.2, random_state=42)

print("X_train shape: ", trainer.X_train.shape)
print("X_val shape: ", trainer.X_val.shape)
print("X_test shape: ", trainer.X_test.shape)

trainer.train()

trainer.test()

X_train shape:  (600, 3)
X_val shape:  (200, 3)
X_test shape:  (200, 3)
Validation loss improved from inf to 255778.7188 in epoch 1. Saving model...
Epoch 1/5000 - Train Loss: 345188.1842, Val Loss: 255778.7188
Validation loss improved from 255778.7188 to 182410.4219 in epoch 2. Saving model...
Validation loss improved from 182410.4219 to 139060.2812 in epoch 3. Saving model...
Validation loss improved from 139060.2812 to 87810.0703 in epoch 4. Saving model...
Validation loss improved from 87810.0703 to 56504.6172 in epoch 5. Saving model...
Validation loss improved from 56504.6172 to 26517.2617 in epoch 6. Saving model...
Validation loss improved from 26517.2617 to 15104.8799 in epoch 7. Saving model...
Validation loss improved from 15104.8799 to 9747.0762 in epoch 8. Saving model...
Validation loss improved from 9747.0762 to 8909.9551 in epoch 9. Saving model...
Validation loss improved from 8909.9551 to 7495.8228 in epoch 10. Saving model...
Validation loss improved from 7495.8228 t

In [4]:
trainer.model

distModel(
  (input_layer): Linear(in_features=3, out_features=50, bias=True)
  (hidden_layers): ModuleList()
  (output_mean_layer): Linear(in_features=50, out_features=1, bias=True)
  (output_sd_layer): Linear(in_features=50, out_features=1, bias=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.05, inplace=False)
)

In [5]:
#trainer.predict(X=trainer.X_test)

In [6]:
trainer.evaluate_gaussian_nll()
trainer.evaluate_standardized_residuals()

Baseline Gaussian NLL Loss: 0.4348
Model Gaussian NLL Loss: -1.1317
KS test against N(0,1): statistic = 0.0732, p-value = 0.2229
✅ The standardized residuals do not significantly differ from a standard normal distribution — model variance estimates appear well-calibrated.
Saved: residual_diagnostics\z_histogram.png
Saved: residual_diagnostics\z_qqplot.png


(np.float64(0.07320890511890416), np.float64(0.22289480793583993))

In [7]:
'''# Inputs
x = np.array([[-1.266313, -0.405118, 0.612491]])

# Extract weights from model
state_dict = trainer.model.state_dict()
w0 = state_dict['input_layer.weight'].cpu().numpy()
b0 = state_dict['input_layer.bias'].cpu().numpy()

w1 = state_dict['hidden_layers.0.weight'].cpu().numpy()
b1 = state_dict['hidden_layers.0.bias'].cpu().numpy()

w_mean = state_dict['output_mean_layer.weight'].cpu().numpy()
b_mean = state_dict['output_mean_layer.bias'].cpu().numpy()

w_sd = state_dict['output_sd_layer.weight'].cpu().numpy()
b_sd = state_dict['output_sd_layer.bias'].cpu().numpy()

# Manual forward pass
def relu(x): return np.maximum(0, x)

h0 = relu(x.dot(w0.T) + b0)
h1 = relu(h0.dot(w1.T) + b1)
mean = h1.dot(w_mean.T) + b_mean
sd = relu(h1.dot(w_sd.T) + b_sd)

print("Manual mean:", mean)
print("Manual std:", sd)'''

'# Inputs\nx = np.array([[-1.266313, -0.405118, 0.612491]])\n\n# Extract weights from model\nstate_dict = trainer.model.state_dict()\nw0 = state_dict[\'input_layer.weight\'].cpu().numpy()\nb0 = state_dict[\'input_layer.bias\'].cpu().numpy()\n\nw1 = state_dict[\'hidden_layers.0.weight\'].cpu().numpy()\nb1 = state_dict[\'hidden_layers.0.bias\'].cpu().numpy()\n\nw_mean = state_dict[\'output_mean_layer.weight\'].cpu().numpy()\nb_mean = state_dict[\'output_mean_layer.bias\'].cpu().numpy()\n\nw_sd = state_dict[\'output_sd_layer.weight\'].cpu().numpy()\nb_sd = state_dict[\'output_sd_layer.bias\'].cpu().numpy()\n\n# Manual forward pass\ndef relu(x): return np.maximum(0, x)\n\nh0 = relu(x.dot(w0.T) + b0)\nh1 = relu(h0.dot(w1.T) + b1)\nmean = h1.dot(w_mean.T) + b_mean\nsd = relu(h1.dot(w_sd.T) + b_sd)\n\nprint("Manual mean:", mean)\nprint("Manual std:", sd)'

In [8]:
cons = trainer.constraint_build()
cons.to_csv('constraints.csv')

In [9]:
cons = pd.read_csv('constraints.csv', index_col=0)
pd.set_option('display.precision', 10)
cons

,intercept,layer,node,node_0,node_1,node_2,node_3,node_4,node_5,node_6,...,node_40,node_41,node_42,node_43,node_44,node_45,node_46,node_47,node_48,node_49
0,-0.2141019700,0,0,0.296281640,-0.2567882500,-0.1232720500,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.7054593000,0,1,0.207522020,-0.3485039000,0.3076981000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,-0.1057791860,0,2,-0.054186980,0.3172796400,0.0428181500,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.3189473700,0,3,-0.119734940,0.0983284800,0.0407778620,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.1962385500,0,4,0.117487700,0.1009395600,-0.7765464000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,-0.3715920000,0,5,-0.051977277,0.0836834300,-0.0023064017,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,-0.3013923800,0,6,0.544596400,0.1449148200,-0.2017014600,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,-0.8214972000,0,7,-0.263572160,-0.1033989040,-0.9038853000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,0.7377847000,0,8,-0.213908420,0.0935788100,0.2366237800,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,-0.1321359600,0,9,0.355235640,-0.5896207700,0.3651097400,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
n_sc =  30
#contextual_var = trainer.X_test
contextual_var = trainer.X_test.iloc[[0]]
contextual_var.reset_index(drop=True, inplace=True)
contextual_var

,x1,x2,price
0,0.1317971241,0.4232970643,-0.6373367167


In [11]:
contextual_var.columns.sort_values()

Index(['price', 'x1', 'x2'], dtype='object')

In [12]:
row = pd.DataFrame([{
    'x1': -1.266313,
    'x2': -0.405118,
    'price': 30.0
}])

def preict_one_row(row):
    # Predict
    mean_pred, sd_pred = trainer.predict(row, unscale=True)
    print("Mean prediction:", mean_pred)
    print("Standard deviation prediction:", sd_pred)

preict_one_row(row)

Mean prediction: [[-14.73813958]]
Standard deviation prediction: [[0.]]


In [13]:
model = pyo.ConcreteModel()

model.N = pyo.RangeSet(0,n_preds-1)
model.W = pyo.RangeSet(1,n_sc) # scenarios

pi = [1/n_sc for i in range(n_sc)]

model.var_ind = pyo.Set(initialize=contextual_var.columns.sort_values())

model.x = pyo.Var(model.var_ind, model.N) # price

model.y = pyo.Var(pyo.Any, dense=False, domain=pyo.Reals)

model.rent = pyo.Var(model.W, within=pyo.Reals) # rentabilidad per scenario

#definition of the objective function
def obj_expression(model):
    return sum(pi[w-1] * model.rent[w] for w in model.W)
model.OBJ = pyo.Objective(rule=obj_expression, sense=pyo.maximize)

#definition of the constraints
def rent_expression(model, w):
    return model.rent[w] == sum(model.y['demand', n, w]*model.x['price',n] for n in model.N)
model.rent_expression = pyo.Constraint(model.W, rule=rent_expression)

#constraint for fixing contextual information
def const9_rule(model,x_ind,n):
    if x_ind=='price':
        return pyo.Constraint.Skip
    else:
        return model.x[x_ind,n] == contextual_var.loc[n,x_ind]
model.const9 = pyo.Constraint(model.var_ind, model.N, rule=const9_rule)

P_max = 30
# max price constraint for each price
def max_price_expression(model, n):
    return model.x['price', n] <= P_max

model.max_price_expression = pyo.Constraint(model.N, rule=max_price_expression)

def min_price_expression(model, n):
    return model.x['price', n] >= 0
model.min_price_expression = pyo.Constraint(model.N, rule=min_price_expression)



In [14]:
const_embed(model=trainer, opt_model=model, constaints=cons, outcome='demand', n_scenarios=n_sc)

In [15]:
opt = pyo.SolverFactory('gurobi_direct')
opt.options['threads'] = 0
opt.options['NonConvex'] = 2
results = opt.solve(model, tee=True)

Set parameter OutputFlag to value 1
Set parameter Threads to value 0
Set parameter NonConvex to value 2
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 13th Gen Intel(R) Core(TM) i5-13500, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Non-default parameters:
NonConvex  2

Optimize a model with 186 rows, 165 columns and 746 nonzeros
Model fingerprint: 0x1d719707
Model has 30 quadratic constraints
Variable types: 115 continuous, 50 integer (50 binary)
Coefficient statistics:
  Matrix range     [4e-04, 1e+03]
  QMatrix range    [1e+00, 1e+00]
  QLMatrix range   [1e+00, 1e+00]
  Objective range  [3e-02, 3e-02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [2e-03, 1e+03]
Presolve removed 58 rows and 68 columns
Presolve time: 0.01s
Presolved: 189 rows, 129 columns, 563 nonzeros
Presolved model has 30 bilinear constraint(s)

Solving non-convex MIQCP

Variable types: 97 contin

In [16]:
for c in model.component_objects(Constraint, active=True):
    print(f"Constraint component: {c.name}")
    for index in c:
        print(f"  {index}: {c[index].expr}")

Constraint component: rent_expression
  1: rent[1]  ==  y[demand,0,1]*x[price,0]
  2: rent[2]  ==  y[demand,0,2]*x[price,0]
  3: rent[3]  ==  y[demand,0,3]*x[price,0]
  4: rent[4]  ==  y[demand,0,4]*x[price,0]
  5: rent[5]  ==  y[demand,0,5]*x[price,0]
  6: rent[6]  ==  y[demand,0,6]*x[price,0]
  7: rent[7]  ==  y[demand,0,7]*x[price,0]
  8: rent[8]  ==  y[demand,0,8]*x[price,0]
  9: rent[9]  ==  y[demand,0,9]*x[price,0]
  10: rent[10]  ==  y[demand,0,10]*x[price,0]
  11: rent[11]  ==  y[demand,0,11]*x[price,0]
  12: rent[12]  ==  y[demand,0,12]*x[price,0]
  13: rent[13]  ==  y[demand,0,13]*x[price,0]
  14: rent[14]  ==  y[demand,0,14]*x[price,0]
  15: rent[15]  ==  y[demand,0,15]*x[price,0]
  16: rent[16]  ==  y[demand,0,16]*x[price,0]
  17: rent[17]  ==  y[demand,0,17]*x[price,0]
  18: rent[18]  ==  y[demand,0,18]*x[price,0]
  19: rent[19]  ==  y[demand,0,19]*x[price,0]
  20: rent[20]  ==  y[demand,0,20]*x[price,0]
  21: rent[21]  ==  y[demand,0,21]*x[price,0]
  22: rent[22]  ==  y[d

In [17]:
value(model.x['price', 0])

17.20544914751156

In [18]:
value(model.y['demand', 0, 1])

4.073265370946168

In [19]:
print(value(model.y['demand', 0, 'mean']))
print(value(model.y['demand', 0, 'sd']))

-0.14326587954820888
0.29234956641930726


In [20]:
opt_profit = value(model.OBJ)
print(f"Optimal profit: {opt_profit}")

Optimal profit: 49.59579283840131


In [21]:
opt_price = value(model.x['price',0])
opt_mean = value(model.y['demand', 0,'mean'])
opt_sd = value(model.y['demand', 0,'sd'])

print(f"Optimal price: {opt_price}")
print(f"Optimal mean: {opt_mean}")
print(f"Optimal sd: {opt_sd}")

Optimal price: 17.20544914751156
Optimal mean: -0.14326587954820888
Optimal sd: 0.29234956641930726
